**chartevents table**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
DATA_PATH = Path('data')
PROCESSED_PATH = Path('data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# 기존 피처 데이터 로드
df_features = pd.read_csv(PROCESSED_PATH / 'final_features_complete.csv')

# 날짜 컬럼 파싱
date_cols = ['admittime', 'dischtime', 'dod']
for col in date_cols:
    if col in df_features.columns:
        df_features[col] = pd.to_datetime(df_features[col])

print(f"✓ 기존 피처 로드 완료")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 컬럼: {len(df_features.columns)}개")
print(f"  - 재입원률: {df_features['readmission_30d'].mean()*100:.2f}%")

# 코호트 정보
cohort_hadm_ids = set(df_features['hadm_id'].unique())
print(f"  - 코호트 입원: {len(cohort_hadm_ids):,}건")

print("\n사용 가능한 데이터 확인")
print("-" * 50)

# 데이터 파일 확인
import os
csv_files = sorted([f for f in os.listdir(DATA_PATH) if f.endswith('.csv')])

key_files = ['chartevents.csv', 'labevents.csv', 'inputevents.csv', 
             'outputevents.csv', 'd_items.csv', 'd_labitems.csv']

print("추가 피처 추출에 필요한 파일:")
for file in key_files:
    if file in csv_files:
        file_path = DATA_PATH / file
        file_size = file_path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {file:<30} {file_size:>8.1f} MB")
    else:
        print(f"  ✗ {file:<30} 없음")

In [ ]:
# Vital Signs 피처 추출 (chartevents)
# d_items 로드
df_items = pd.read_csv(DATA_PATH / 'd_items.csv')
print(f"총 chart 항목: {len(df_items):,}개")

# 중요 바이탈 항목 정의
important_vitals = {
    'Heart Rate': ['heart rate', 'hr ', 'pulse'],
    'Blood Pressure - Systolic': ['systolic', 'sbp', 'arterial bp systolic', 'nbp systolic'],
    'Blood Pressure - Diastolic': ['diastolic', 'dbp', 'arterial bp diastolic', 'nbp diastolic'],
    'Blood Pressure - Mean': ['mean arterial', 'map', 'arterial bp mean', 'nbp mean'],
    'Respiratory Rate': ['respiratory rate', 'resp rate', 'rr '],
    'SpO2': ['spo2', 'o2 saturation', 'oxygen saturation'],
    'Temperature': ['temperature', 'temp '],
    'Weight': ['weight', 'admit wt', 'daily weight'],
    'Height': ['height'],
    'O2 Flow': ['o2 flow', 'oxygen flow'],
    'FiO2': ['fio2', 'inspired o2']
}

# 각 카테고리별 itemid 찾기
vital_itemids = {}

for vital_name, keywords in important_vitals.items():
    pattern = '|'.join(keywords)
    mask = df_items['label'].str.lower().str.contains(pattern, regex=True, na=False)
    matched = df_items[mask]
    
    if len(matched) > 0:
        vital_itemids[vital_name] = matched['itemid'].tolist()
        print(f"\n{vital_name}: {len(matched)}개 항목")
        for idx, row in matched.head(5).iterrows():
            print(f"  - {row['itemid']}: {row['label']}")
        if len(matched) > 5:
            print(f"  ... 외 {len(matched)-5}개")
    else:
        print(f"\n{vital_name}: 찾을 수 없음")

# 선택된 itemid
all_itemids = []
for itemids in vital_itemids.values():
    all_itemids.extend(itemids)
selected_itemids = set(all_itemids)

print(f"\n" + "=" * 80)
print(f"총 선택된 항목: {len(selected_itemids)}개")
print(f"카테고리: {len(vital_itemids)}개")

In [ ]:
# chartevents 데이터 추출
print(f"코호트 입원: {len(cohort_hadm_ids):,}건")
print(f"추출할 항목: {len(selected_itemids)}개")

print(f"\n청크 처리 시작...")
print(f"  - 파일 크기: 40.0 GB")
print(f"  - 예상 시간: 1-2시간")
print(f"  - 청크 크기: 1,000,000 rows")

vital_data = []
chunk_size = 1000000
processed_rows = 0
found_rows = 0

start_time = datetime.now()
last_print_time = start_time

try:
    for chunk_num, chunk in enumerate(pd.read_csv(DATA_PATH / 'chartevents.csv', 
                                                    chunksize=chunk_size,
                                                    dtype={'value': str}), 1):
        
        # 필터링
        chunk_filtered = chunk[chunk['hadm_id'].isin(cohort_hadm_ids)]
        chunk_filtered = chunk_filtered[chunk_filtered['itemid'].isin(selected_itemids)]
        
        if len(chunk_filtered) > 0:
            vital_data.append(chunk_filtered)
            found_rows += len(chunk_filtered)
        
        processed_rows += len(chunk)
        
        # 진행 상황 출력
        current_time = datetime.now()
        time_since_print = (current_time - last_print_time).total_seconds()
        
        if chunk_num % 10 == 0 or time_since_print > 300:
            elapsed = (current_time - start_time).total_seconds()
            rate = processed_rows / elapsed if elapsed > 0 else 0
            
            print(f"  [{current_time.strftime('%H:%M:%S')}] "
                  f"처리: {processed_rows:,} rows ({chunk_num} chunks) "
                  f"| 발견: {found_rows:,} rows "
                  f"| 속도: {rate:,.0f} rows/sec")
            
            last_print_time = current_time
    
    print(f"\n✓ 청크 처리 완료")
    
    if len(vital_data) > 0:
        print(f"  병합 중...")
        df_chart_cohort = pd.concat(vital_data, ignore_index=True)
        
        print(f"  - 추출: {len(df_chart_cohort):,}행")
        print(f"  - 고유 입원: {df_chart_cohort['hadm_id'].nunique():,}건")
        print(f"  - 고유 항목: {df_chart_cohort['itemid'].nunique():,}개")
        
        # 임시 저장
        temp_file = PROCESSED_PATH / 'temp_chartevents_extracted.csv'
        df_chart_cohort.to_csv(temp_file, index=False)
        print(f"  - 임시 저장: {temp_file}")
        
        del vital_data
        import gc
        gc.collect()
    else:
        df_chart_cohort = pd.DataFrame()
        
except KeyboardInterrupt:
    print("\n중단됨")
    if len(vital_data) > 0:
        df_chart_cohort = pd.concat(vital_data, ignore_index=True)
        print(f"  ✓ 부분 데이터: {len(df_chart_cohort):,}행")
    else:
        df_chart_cohort = pd.DataFrame()
        
except Exception as e:
    print(f"\n오류: {str(e)}")

elapsed_total = (datetime.now() - start_time).total_seconds()
print(f"\n총 소요 시간: {elapsed_total/60:.1f}분")

In [ ]:
# 추출된 데이터 확인
print(f"추출된 데이터:")
print(f"  - 총 측정값: {len(df_chart_cohort):,}개")
print(f"  - 고유 입원: {df_chart_cohort['hadm_id'].nunique():,}건")
print(f"  - 고유 환자: {df_chart_cohort['subject_id'].nunique():,}명")
print(f"  - 고유 항목: {df_chart_cohort['itemid'].nunique():,}개")

print(f"\n데이터 구조:")
print(df_chart_cohort.head(10))

print(f"\n컬럼:")
print(df_chart_cohort.columns.tolist())

print(f"\n데이터 타입:")
print(df_chart_cohort.dtypes)

# 날짜 컬럼 파싱
if 'charttime' in df_chart_cohort.columns:
    df_chart_cohort['charttime'] = pd.to_datetime(df_chart_cohort['charttime'])
    print(f"\n✓ charttime 파싱 완료")

# value 컬럼을 숫자로 변환
df_chart_cohort['valuenum'] = pd.to_numeric(df_chart_cohort['value'], errors='coerce')
print(f"\n✓ valuenum 생성 완료")

# 결측치 확인
null_count = df_chart_cohort['valuenum'].isnull().sum()
null_pct = null_count / len(df_chart_cohort) * 100
print(f"  - valuenum 결측: {null_count:,}개 ({null_pct:.1f}%)")

# 항목별 분포
print(f"\n항목별 측정 횟수 (상위 20개):")
item_counts = df_chart_cohort['itemid'].value_counts().head(20)
for itemid, count in item_counts.items():
    # item 이름 찾기
    item_name = df_items[df_items['itemid'] == itemid]['label'].values
    if len(item_name) > 0:
        print(f"  - {itemid} ({item_name[0][:40]}): {count:,}회")
    else:
        print(f"  - {itemid}: {count:,}회")

In [ ]:
# 입원별 Vital Signs 집계
print("각 입원의 첫 24시간 측정값 집계")
print("  - 첫 측정값 (First)")
print("  - 최솟값 (Min)")
print("  - 최댓값 (Max)")
print("  - 평균 (Mean)")

# 입원 시간 정보 추가
df_chart_cohort = df_chart_cohort.merge(
    df_features[['hadm_id', 'admittime']],
    on='hadm_id',
    how='left'
)

# 입원 후 시간 계산
df_chart_cohort['hours_from_admit'] = (
    df_chart_cohort['charttime'] - df_chart_cohort['admittime']
).dt.total_seconds() / 3600

# 첫 24시간만 (입원 초기 상태가 가장 중요)
df_chart_24h = df_chart_cohort[
    (df_chart_cohort['hours_from_admit'] >= 0) & 
    (df_chart_cohort['hours_from_admit'] <= 24)
].copy()

print(f"\n첫 24시간 측정값:")
print(f"  - 전체: {len(df_chart_cohort):,}개")
print(f"  - 첫 24시간: {len(df_chart_24h):,}개 ({len(df_chart_24h)/len(df_chart_cohort)*100:.1f}%)")

# itemid별로 집계
vital_features_list = []

for vital_name, itemids in vital_itemids.items():
    print(f"\n{vital_name} 집계 중...")
    
    # 해당 항목의 데이터만
    vital_data = df_chart_24h[df_chart_24h['itemid'].isin(itemids)].copy()
    
    if len(vital_data) == 0:
        print(f"데이터 없음")
        continue
    
    # 입원별 집계
    vital_agg = vital_data.groupby('hadm_id')['valuenum'].agg([
        ('first', 'first'),
        ('min', 'min'),
        ('max', 'max'),
        ('mean', 'mean')
    ]).reset_index()
    
    # 컬럼명 변경 (공백 제거, 소문자)
    prefix = vital_name.lower().replace(' ', '_').replace('-', '_')
    vital_agg.columns = [
        'hadm_id',
        f'{prefix}_first',
        f'{prefix}_min',
        f'{prefix}_max',
        f'{prefix}_mean'
    ]
    
    print(f"  - 입원 수: {len(vital_agg):,}건")
    print(f"  - 평균 측정 횟수: {len(vital_data)/len(vital_agg):.1f}회")
    
    vital_features_list.append(vital_agg)

# 모든 vital signs 병합
print(f"\n모든 vital signs 병합 중...")
df_vital_features = vital_features_list[0]
for vital_df in vital_features_list[1:]:
    df_vital_features = df_vital_features.merge(vital_df, on='hadm_id', how='outer')

print(f"\n✓ Vital Signs 피처 생성 완료")
print(f"  - 입원 수: {len(df_vital_features):,}건")
print(f"  - 피처 수: {len(df_vital_features.columns)-1}개 (hadm_id 제외)")

# 피처 목록
feature_cols = [col for col in df_vital_features.columns if col != 'hadm_id']
print(f"\n생성된 피처:")
for col in sorted(feature_cols):
    non_null = df_vital_features[col].notna().sum()
    print(f"  - {col:<50} {non_null:>6,}건 ({non_null/len(df_vital_features)*100:>5.1f}%)")

In [ ]:
# 추가 파생 피처 생성
# Pulse Pressure (맥압) = SBP - DBP
if 'blood_pressure___systolic_mean' in df_vital_features.columns and \
   'blood_pressure___diastolic_mean' in df_vital_features.columns:
    
    df_vital_features['pulse_pressure'] = (
        df_vital_features['blood_pressure___systolic_mean'] - 
        df_vital_features['blood_pressure___diastolic_mean']
    )
    print("✓ Pulse Pressure 생성")

# Shock Index (쇼크 지수) = HR / SBP
if 'heart_rate_mean' in df_vital_features.columns and \
   'blood_pressure___systolic_mean' in df_vital_features.columns:
    
    df_vital_features['shock_index'] = (
        df_vital_features['heart_rate_mean'] / 
        df_vital_features['blood_pressure___systolic_mean']
    )
    
    # Shock Index 범주화
    df_vital_features['shock_index_high'] = (
        df_vital_features['shock_index'] > 0.9
    ).astype(int)
    
    print("✓ Shock Index 생성")
    print(f"  - Shock Index > 0.9: {df_vital_features['shock_index_high'].sum():,}건")

# Hypotension (저혈압) - SBP < 90
if 'blood_pressure___systolic_min' in df_vital_features.columns:
    df_vital_features['hypotension'] = (
        df_vital_features['blood_pressure___systolic_min'] < 90
    ).astype(int)
    print("✓ Hypotension 피처 생성")
    print(f"  - 저혈압 경험: {df_vital_features['hypotension'].sum():,}건")

# Tachycardia (빈맥) - HR > 100
if 'heart_rate_max' in df_vital_features.columns:
    df_vital_features['tachycardia'] = (
        df_vital_features['heart_rate_max'] > 100
    ).astype(int)
    print("✓ Tachycardia 피처 생성")
    print(f"  - 빈맥 경험: {df_vital_features['tachycardia'].sum():,}건")

# Bradycardia (서맥) - HR < 60
if 'heart_rate_min' in df_vital_features.columns:
    df_vital_features['bradycardia'] = (
        df_vital_features['heart_rate_min'] < 60
    ).astype(int)
    print("✓ Bradycardia 피처 생성")
    print(f"  - 서맥 경험: {df_vital_features['bradycardia'].sum():,}건")

# Hypoxemia (저산소증) - SpO2 < 90
if 'spo2_min' in df_vital_features.columns:
    df_vital_features['hypoxemia'] = (
        df_vital_features['spo2_min'] < 90
    ).astype(int)
    print("✓ Hypoxemia 피처 생성")
    print(f"  - 저산소증 경험: {df_vital_features['hypoxemia'].sum():,}건")

# Tachypnea (빈호흡) - RR > 20
if 'respiratory_rate_max' in df_vital_features.columns:
    df_vital_features['tachypnea'] = (
        df_vital_features['respiratory_rate_max'] > 20
    ).astype(int)
    print("✓ Tachypnea 피처 생성")
    print(f"  - 빈호흡 경험: {df_vital_features['tachypnea'].sum():,}건")

print(f"\n총 생성된 피처: {len(df_vital_features.columns)-1}개")

In [ ]:
# 기존 피처에 Vital Signs 병합
print(f"병합 전:")
print(f"  - 기존 피처: {len(df_features.columns)}개")
print(f"  - Vital Signs 피처: {len(df_vital_features.columns)-1}개")

# 병합
df_features = df_features.merge(df_vital_features, on='hadm_id', how='left')

print(f"\n병합 후:")
print(f"  - 총 피처: {len(df_features.columns)}개")
print(f"  - 추가된 피처: {len(df_vital_features.columns)-1}개")

# Vital Signs 없는 입원
no_vitals = df_features['heart_rate_mean'].isnull().sum()
print(f"\nVital Signs 없는 입원: {no_vitals:,}건 ({no_vitals/len(df_features)*100:.1f}%)")
print(f"  → ICU 입실 안 한 환자들 (일반 병동)")

# 결측치 처리 전략:
# 연속형 피처: 결측 유지 (모델에서 처리)
# 이진형 피처 (hypotension 등): 0으로 대체 (0 = 정상)

# 이진형 피처만 0으로 대체
binary_features = ['shock_index_high', 'hypotension', 'tachycardia', 
                   'bradycardia', 'hypoxemia', 'tachypnea']
for col in binary_features:
    if col in df_features.columns:
        df_features[col].fillna(0, inplace=True)
        print(f"  ✓ {col}: 결측 → 0")

print(f"\n✓ Vital Signs 피처 병합 완료!")

In [ ]:
# Vital Signs 피처 저장
output_file = PROCESSED_PATH / 'features_after_vitals.csv'
df_features.to_csv(output_file, index=False)

file_size = output_file.stat().st_size / (1024 * 1024)
print(f"✓ 저장 완료: {output_file}")
print(f"  - 크기: {file_size:.1f} MB")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 컬럼: {len(df_features.columns)}개")

print(f"\n추가된 피처 카테고리:")
vital_feature_cols = [col for col in df_features.columns if col in df_vital_features.columns and col != 'hadm_id']
print(f"  - Vital Signs 원시 피처: {len([c for c in vital_feature_cols if any(x in c for x in ['_first', '_min', '_max', '_mean'])])}개")
print(f"  - 파생 피처: {len([c for c in vital_feature_cols if c in binary_features + ['pulse_pressure', 'shock_index']])}개")

**labevents table**

In [ ]:
# Lab Values 피처 추출 (labevents)
# d_labitems 로드 (Lab 항목 정보)
df_labitems = pd.read_csv(DATA_PATH / 'd_labitems.csv')
print(f"총 Lab 항목: {len(df_labitems):,}개")

print(f"\n컬럼: {list(df_labitems.columns)}")
print(f"\n샘플:")
print(df_labitems.head(10))

# 심부전 관련 주요 Lab 항목
important_labs = {
    # 심부전 마커 (가장 중요)
    'BNP': ['bnp', 'b-type natriuretic', 'brain natriuretic', 'pro-bnp'],
    'NT-proBNP': ['nt-probnp', 'nt probnp', 'n-terminal pro', 'proBNP'],
    'Troponin': ['troponin'],
    
    # 신기능 (심부전과 밀접)
    'Creatinine': ['creatinine'],
    'BUN': ['urea nitrogen', 'bun'],
    
    # 전해질 (심부전 관리 핵심)
    'Sodium': ['sodium'],
    'Potassium': ['potassium'],
    'Chloride': ['chloride'],
    'Bicarbonate': ['bicarbonate', 'co2'],
    'Magnesium': ['magnesium'],
    'Calcium': ['calcium'],
    'Phosphate': ['phosphate', 'phosphorus'],
    
    # 혈액학 (빈혈, 감염)
    'Hemoglobin': ['hemoglobin', 'hgb', 'hb '],
    'Hematocrit': ['hematocrit', 'hct'],
    'WBC': ['white blood', 'wbc', 'leukocyte'],
    'Platelet': ['platelet'],
    
    # 간기능/영양
    'Albumin': ['albumin'],
    'Bilirubin': ['bilirubin'],
    'ALT': ['alanine aminotransferase', 'alt', 'sgpt'],
    'AST': ['aspartate aminotransferase', 'ast', 'sgot'],
    
    # 기타
    'Glucose': ['glucose'],
    'Lactate': ['lactate']
}

# 각 카테고리별 itemid 찾기
lab_itemids = {}

for lab_name, keywords in important_labs.items():
    pattern = '|'.join(keywords)
    mask = df_labitems['label'].str.lower().str.contains(pattern, regex=True, na=False)
    matched = df_labitems[mask]
    
    if len(matched) > 0:
        lab_itemids[lab_name] = matched['itemid'].tolist()
        print(f"\n{lab_name}: {len(matched)}개 항목")
        for idx, row in matched.head(5).iterrows():
            print(f"  - {row['itemid']}: {row['label']}")
        if len(matched) > 5:
            print(f"  ... 외 {len(matched)-5}개")
    else:
        print(f"\n{lab_name}: 찾을 수 없음")

# 선택된 itemid
all_itemids = []
for itemids in lab_itemids.values():
    all_itemids.extend(itemids)
selected_itemids = set(all_itemids)

print(f"\n" + "=" * 80)
print(f"총 선택된 Lab 항목: {len(selected_itemids)}개")
print(f"카테고리: {len(lab_itemids)}개")

In [ ]:
# labevents 데이터 추출
print(f"코호트 입원: {len(cohort_hadm_ids):,}건")
print(f"추출할 Lab 항목: {len(selected_itemids)}개")

print(f"\n청크 처리 시작")
print(f"  - 청크 크기: 1,000,000 rows")

lab_data = []
chunk_size = 1000000
processed_rows = 0
found_rows = 0

start_time = datetime.now()
last_print_time = start_time

try:
    for chunk_num, chunk in enumerate(pd.read_csv(DATA_PATH / 'labevents.csv', 
                                                    chunksize=chunk_size), 1):
        
        # 필터링
        chunk_filtered = chunk[chunk['hadm_id'].isin(cohort_hadm_ids)]
        chunk_filtered = chunk_filtered[chunk_filtered['itemid'].isin(selected_itemids)]
        
        if len(chunk_filtered) > 0:
            lab_data.append(chunk_filtered)
            found_rows += len(chunk_filtered)
        
        processed_rows += len(chunk)
        
        # 진행 상황 출력 (10 청크마다 또는 5분마다)
        current_time = datetime.now()
        time_since_print = (current_time - last_print_time).total_seconds()
        
        if chunk_num % 10 == 0 or time_since_print > 300:
            elapsed = (current_time - start_time).total_seconds()
            rate = processed_rows / elapsed if elapsed > 0 else 0
            
            print(f"  [{current_time.strftime('%H:%M:%S')}] "
                  f"처리: {processed_rows:,} rows ({chunk_num} chunks) "
                  f"| 발견: {found_rows:,} rows "
                  f"| 속도: {rate:,.0f} rows/sec")
            
            last_print_time = current_time
    
    print(f"\n✓ 청크 처리 완료")
    
    if len(lab_data) > 0:
        print(f"  병합 중...")
        df_lab_cohort = pd.concat(lab_data, ignore_index=True)
        
        print(f"  - 추출: {len(df_lab_cohort):,}행")
        print(f"  - 고유 입원: {df_lab_cohort['hadm_id'].nunique():,}건")
        print(f"  - 고유 환자: {df_lab_cohort['subject_id'].nunique():,}명")
        print(f"  - 고유 항목: {df_lab_cohort['itemid'].nunique():,}개")
        
        # 임시 저장
        temp_file = PROCESSED_PATH / 'temp_labevents_extracted.csv'
        df_lab_cohort.to_csv(temp_file, index=False)
        print(f"  - 임시 저장: {temp_file}")
        
        del lab_data
        import gc
        gc.collect()
    else:
        print("추출된 데이터 없음")
        df_lab_cohort = pd.DataFrame()
        
except KeyboardInterrupt:
    print("\n중단됨")
    if len(lab_data) > 0:
        df_lab_cohort = pd.concat(lab_data, ignore_index=True)
        print(f"  ✓ 부분 데이터: {len(df_lab_cohort):,}행")
    else:
        df_lab_cohort = pd.DataFrame()
        
except Exception as e:
    print(f"\n오류: {str(e)}")
    import traceback
    traceback.print_exc()

elapsed_total = (datetime.now() - start_time).total_seconds()
print(f"\n총 소요 시간: {elapsed_total/60:.1f}분")

In [ ]:
# 추출된 Lab 데이터 확인
print(f"추출된 데이터:")
print(f"  - 총 측정값: {len(df_lab_cohort):,}개")
print(f"  - 고유 입원: {df_lab_cohort['hadm_id'].nunique():,}건")
print(f"  - 고유 환자: {df_lab_cohort['subject_id'].nunique():,}명")
print(f"  - 고유 항목: {df_lab_cohort['itemid'].nunique():,}개")

print(f"\n데이터 구조:")
print(df_lab_cohort.head(10))

print(f"\n컬럼:")
print(df_lab_cohort.columns.tolist())

print(f"\n데이터 타입:")
print(df_lab_cohort.dtypes)

# 날짜 파싱
if 'charttime' in df_lab_cohort.columns:
    df_lab_cohort['charttime'] = pd.to_datetime(df_lab_cohort['charttime'])
    print(f"\n✓ charttime 파싱 완료")

# valuenum 결측 확인
if 'valuenum' in df_lab_cohort.columns:
    null_count = df_lab_cohort['valuenum'].isnull().sum()
    null_pct = null_count / len(df_lab_cohort) * 100
    print(f"\nvaluenum 결측: {null_count:,}개 ({null_pct:.1f}%)")
else:
    print(f"\nvaluenum 컬럼 없음 - 생성 필요")

# 항목별 측정 횟수 (상위 20개)
print(f"\n항목별 측정 횟수 (상위 20개):")
item_counts = df_lab_cohort['itemid'].value_counts().head(20)
for itemid, count in item_counts.items():
    # item 이름 찾기
    item_name = df_labitems[df_labitems['itemid'] == itemid]['label'].values
    if len(item_name) > 0:
        print(f"  - {itemid} ({item_name[0][:50]}): {count:,}회")
    else:
        print(f"  - {itemid}: {count:,}회")

# 카테고리별 커버리지
print(f"\n카테고리별 입원 커버리지:")
for lab_name, itemids in lab_itemids.items():
    lab_hadm = df_lab_cohort[df_lab_cohort['itemid'].isin(itemids)]['hadm_id'].nunique()
    coverage = lab_hadm / len(cohort_hadm_ids) * 100
    print(f"  - {lab_name:<15}: {lab_hadm:>6,}건 ({coverage:>5.1f}%)")

In [ ]:
# 입원별 Lab Values 집계
print("입원 후 7일 이내 측정값 집계 중")
print("  - 첫 측정값 (First)")
print("  - 최솟값 (Min)")
print("  - 최댓값 (Max)")
print("  - 평균 (Mean)")

# 입원 시간 정보 추가
df_lab_cohort = df_lab_cohort.merge(
    df_features[['hadm_id', 'admittime']],
    on='hadm_id',
    how='left'
)

# 입원 후 시간 계산
df_lab_cohort['days_from_admit'] = (
    df_lab_cohort['charttime'] - df_lab_cohort['admittime']
).dt.total_seconds() / (3600 * 24)

# 입원 후 7일 이내
df_lab_7d = df_lab_cohort[
    (df_lab_cohort['days_from_admit'] >= 0) & 
    (df_lab_cohort['days_from_admit'] <= 7)
].copy()

print(f"\n입원 후 7일 이내 측정값:")
print(f"  - 전체: {len(df_lab_cohort):,}개")
print(f"  - 7일 이내: {len(df_lab_7d):,}개 ({len(df_lab_7d)/len(df_lab_cohort)*100:.1f}%)")

# 이상치 제거 함수 (극단적 값 필터링/ 측정 오류 가능성 有)
def remove_outliers(df, column, lower_percentile=0.1, upper_percentile=99.9):
    lower = df[column].quantile(lower_percentile / 100)
    upper = df[column].quantile(upper_percentile / 100)
    return df[(df[column] >= lower) & (df[column] <= upper)]

# 카테고리별 집계
lab_features_list = []

for lab_name, itemids in lab_itemids.items():
    print(f"\n{lab_name} 집계 중...")
    
    # 해당 항목의 데이터만
    lab_data = df_lab_7d[df_lab_7d['itemid'].isin(itemids)].copy()
    
    if len(lab_data) == 0:
        print(f"데이터 없음")
        continue
    
    # 이상치 제거 (valuenum 기준)
    if 'valuenum' in lab_data.columns:
        original_len = len(lab_data)
        lab_data = remove_outliers(lab_data, 'valuenum')
        removed = original_len - len(lab_data)
        if removed > 0:
            print(f"  - 이상치 제거: {removed:,}개 ({removed/original_len*100:.1f}%)")
    
    # 입원별 집계
    lab_agg = lab_data.groupby('hadm_id')['valuenum'].agg([
        ('first', 'first'),
        ('min', 'min'),
        ('max', 'max'),
        ('mean', 'mean')
    ]).reset_index()
    
    # 컬럼명 변경
    prefix = lab_name.lower().replace('-', '_').replace(' ', '_')
    lab_agg.columns = [
        'hadm_id',
        f'lab_{prefix}_first',
        f'lab_{prefix}_min',
        f'lab_{prefix}_max',
        f'lab_{prefix}_mean'
    ]
    
    print(f"  - 입원 수: {len(lab_agg):,}건")
    print(f"  - 평균 측정 횟수: {len(lab_data)/len(lab_agg):.1f}회")
    print(f"  - 평균값: {lab_agg[f'lab_{prefix}_mean'].mean():.2f}")
    print(f"  - 중앙값: {lab_agg[f'lab_{prefix}_mean'].median():.2f}")
    
    lab_features_list.append(lab_agg)

# 모든 lab 병합
print(f"\n모든 Lab Values 병합")
df_lab_features = lab_features_list[0]
for lab_df in lab_features_list[1:]:
    df_lab_features = df_lab_features.merge(lab_df, on='hadm_id', how='outer')

print(f"\n✓ Lab Values 피처 생성 완료")
print(f"  - 입원 수: {len(df_lab_features):,}건")
print(f"  - 피처 수: {len(df_lab_features.columns)-1}개 (hadm_id 제외)")

# 피처 목록
feature_cols = [col for col in df_lab_features.columns if col != 'hadm_id']
print(f"\n생성된 피처 (상위 30개):")
for col in sorted(feature_cols)[:30]:
    non_null = df_lab_features[col].notna().sum()
    print(f"  - {col:<50} {non_null:>6,}건 ({non_null/len(df_lab_features)*100:>5.1f}%)")

if len(feature_cols) > 30:
    print(f"  ... 외 {len(feature_cols)-30}개")

In [ ]:
# Lab 파생 피처 생성
# eGFR 계산
if 'lab_creatinine_mean' in df_lab_features.columns:
    print("eGFR (신기능) 계산 중...")
    
    # 나이와 성별 정보 병합
    df_lab_features = df_lab_features.merge(
        df_features[['hadm_id', 'age_at_admission', 'gender_male']],
        on='hadm_id',
        how='left'
    )
    
    # CKD-EPI 공식
    # eGFR = 186 × (Cr)^(-1.154) × (Age)^(-0.203) × (0.742 if female)
    df_lab_features['egfr'] = (
        186 * 
        (df_lab_features['lab_creatinine_mean'] ** -1.154) * 
        (df_lab_features['age_at_admission'] ** -0.203) *
        (0.742 + 0.258 * df_lab_features['gender_male'])
    )
    
    # eGFR 범주화
    df_lab_features['egfr_stage_1'] = (df_lab_features['egfr'] >= 90).astype(int)  # 정상
    df_lab_features['egfr_stage_2'] = ((df_lab_features['egfr'] >= 60) & (df_lab_features['egfr'] < 90)).astype(int)  # 경증
    df_lab_features['egfr_stage_3'] = ((df_lab_features['egfr'] >= 30) & (df_lab_features['egfr'] < 60)).astype(int)  # 중등도
    df_lab_features['egfr_stage_4_5'] = (df_lab_features['egfr'] < 30).astype(int)  # 중증/말기
    
    print(f"  ✓ eGFR 계산 완료")
    print(f"    - Stage 1 (≥90): {df_lab_features['egfr_stage_1'].sum():,}건")
    print(f"    - Stage 2 (60-89): {df_lab_features['egfr_stage_2'].sum():,}건")
    print(f"    - Stage 3 (30-59): {df_lab_features['egfr_stage_3'].sum():,}건")
    print(f"    - Stage 4-5 (<30): {df_lab_features['egfr_stage_4_5'].sum():,}건")

# BUN/Creatinine 비율
if 'lab_bun_mean' in df_lab_features.columns and 'lab_creatinine_mean' in df_lab_features.columns:
    df_lab_features['bun_cr_ratio'] = (
        df_lab_features['lab_bun_mean'] / df_lab_features['lab_creatinine_mean']
    )
    print(f"\n✓ BUN/Cr 비율 생성")
    print(f"  - 평균: {df_lab_features['bun_cr_ratio'].mean():.1f}")

# 저나트륨혈증 (Hyponatremia)
if 'lab_sodium_mean' in df_lab_features.columns:
    df_lab_features['hyponatremia'] = (
        df_lab_features['lab_sodium_mean'] < 135
    ).astype(int)
    print(f"\n✓ 저나트륨혈증 피처")
    print(f"  - Na <135: {df_lab_features['hyponatremia'].sum():,}건")

# 저칼륨혈증 / 고칼륨혈증
if 'lab_potassium_mean' in df_lab_features.columns:
    df_lab_features['hypokalemia'] = (
        df_lab_features['lab_potassium_mean'] < 3.5
    ).astype(int)
    df_lab_features['hyperkalemia'] = (
        df_lab_features['lab_potassium_mean'] > 5.0
    ).astype(int)
    print(f"\n✓ 칼륨 이상")
    print(f"  - 저칼륨 (K <3.5): {df_lab_features['hypokalemia'].sum():,}건")
    print(f"  - 고칼륨 (K >5.0): {df_lab_features['hyperkalemia'].sum():,}건")

# 빈혈
if 'lab_hemoglobin_mean' in df_lab_features.columns:
    # 성별별 기준 다름
    df_lab_features['anemia'] = (
        ((df_lab_features['gender_male'] == 1) & (df_lab_features['lab_hemoglobin_mean'] < 13)) |
        ((df_lab_features['gender_male'] == 0) & (df_lab_features['lab_hemoglobin_mean'] < 12))
    ).astype(int)
    print(f"\n✓ 빈혈 피처")
    print(f"  - 빈혈: {df_lab_features['anemia'].sum():,}건")

# 백혈구 증가증 (감염/염증)
if 'lab_wbc_mean' in df_lab_features.columns:
    df_lab_features['leukocytosis'] = (
        df_lab_features['lab_wbc_mean'] > 11
    ).astype(int)
    print(f"\n✓ 백혈구증가증")
    print(f"  - WBC >11: {df_lab_features['leukocytosis'].sum():,}건")

# BNP 고위험군 (심부전 중증도)
if 'lab_bnp_mean' in df_lab_features.columns:
    df_lab_features['bnp_very_high'] = (
        df_lab_features['lab_bnp_mean'] > 400
    ).astype(int)
    print(f"\n✓ BNP 고위험")
    print(f"  - BNP >400: {df_lab_features['bnp_very_high'].sum():,}건")

# NT-proBNP 고위험군
if 'lab_nt_probnp_mean' in df_lab_features.columns:
    df_lab_features['ntprobnp_very_high'] = (
        df_lab_features['lab_nt_probnp_mean'] > 900
    ).astype(int)
    print(f"\n✓ NT-proBNP 고위험")
    print(f"  - NT-proBNP >900: {df_lab_features['ntprobnp_very_high'].sum():,}건")

# 혈당 이상
if 'lab_glucose_mean' in df_lab_features.columns:
    df_lab_features['hyperglycemia'] = (
        df_lab_features['lab_glucose_mean'] > 180
    ).astype(int)
    df_lab_features['hypoglycemia'] = (
        df_lab_features['lab_glucose_mean'] < 70
    ).astype(int)
    print(f"\n✓ 혈당 이상")
    print(f"  - 고혈당 (>180): {df_lab_features['hyperglycemia'].sum():,}건")
    print(f"  - 저혈당 (<70): {df_lab_features['hypoglycemia'].sum():,}건")

# 젖산 상승 (쇼크/저관류)
if 'lab_lactate_mean' in df_lab_features.columns:
    df_lab_features['hyperlactatemia'] = (
        df_lab_features['lab_lactate_mean'] > 2.0
    ).astype(int)
    print(f"\n✓ 젖산 상승")
    print(f"  - Lactate >2.0: {df_lab_features['hyperlactatemia'].sum():,}건")

print(f"\n총 생성된 피처: {len(df_lab_features.columns)-1}개")

In [ ]:
# 기존 피처에 Lab Values 병합
print(f"병합 전:")
print(f"  - 기존 피처: {len(df_features.columns)}개")
print(f"  - Lab 피처: {len(df_lab_features.columns)-1}개")

# age_at_admission, gender_male 중복 제거
lab_cols_to_merge = [col for col in df_lab_features.columns 
                     if col not in ['hadm_id', 'age_at_admission', 'gender_male']]

df_lab_features_clean = df_lab_features[['hadm_id'] + lab_cols_to_merge]

# 병합
df_features = df_features.merge(df_lab_features_clean, on='hadm_id', how='left')

print(f"\n병합 후:")
print(f"  - 총 피처: {len(df_features.columns)}개")
print(f"  - 추가된 피처: {len(lab_cols_to_merge)}개")

# Lab 없는 입원
if 'lab_creatinine_mean' in df_features.columns:
    no_labs = df_features['lab_creatinine_mean'].isnull().sum()
    print(f"\nLab Values 없는 입원: {no_labs:,}건 ({no_labs/len(df_features)*100:.1f}%)")
    print(f"  → 매우 짧은 입원 or 기록 누락")

print(f"\n✓ Lab Values 피처 병합 완료")

In [ ]:
# Lab Values 피처 저장
output_file = PROCESSED_PATH / 'features_after_labs.csv'
df_features.to_csv(output_file, index=False)

file_size = output_file.stat().st_size / (1024 * 1024)
print(f"✓ 저장 완료: {output_file}")
print(f"  - 크기: {file_size:.1f} MB")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 컬럼: {len(df_features.columns)}개")

print("\n" + "=" * 80)
print("Part 9 (Lab Values) 완료!")
print("=" * 80)

# 추가된 피처 분석
lab_feature_cols = [col for col in df_features.columns 
                    if col.startswith('lab_') or col in ['egfr', 'bun_cr_ratio', 
                                                          'hyponatremia', 'hypokalemia', 
                                                          'hyperkalemia', 'anemia',
                                                          'leukocytosis', 'bnp_very_high',
                                                          'ntprobnp_very_high', 'hyperglycemia',
                                                          'hypoglycemia', 'hyperlactatemia',
                                                          'egfr_stage_1', 'egfr_stage_2',
                                                          'egfr_stage_3', 'egfr_stage_4_5']]

print(f"\n추가된 Lab 피처: {len(lab_feature_cols)}개")

print(f"\n전체 피처 현황:")
print(f"  - Part 1-7 (기본): 114개")
print(f"  - Part 8 (Vitals): 48개")
print(f"  - Part 9 (Labs): {len(lab_feature_cols)}개")
print(f"  - 총: {len(df_features.columns)}개")

create dataset

In [ ]:
# 최종 모델링 데이터셋 저장
# 현재 상태 확인
print("\n현재 데이터 상태:")
print(f"  - 총 행: {len(df_features):,}건")
print(f"  - 총 컬럼: {len(df_features.columns)}개")
print(f"  - 재입원률: {df_features['readmission_30d'].mean()*100:.2f}%")

# 컬럼 분류
print("\n" + "-" * 80)
print("컬럼 분류")
print("-" * 80)

# ID 컬럼
id_cols = ['subject_id', 'hadm_id']

# 날짜 컬럼
date_cols = ['admittime', 'dischtime', 'dod']

# Target 컬럼
target_col = 'readmission_30d'

# 보조 정보 (모델링 사용 X)
auxiliary_cols = [
    'days_to_readmission',
    'hospital_expire_flag',
    'admission_type',
    'discharge_location',
    'anchor_age',
    'anchor_year',
    'anchor_year_group',
    'gender',
    'age_group',
    'admit_year'
]

# 실제 존재하는 보조 컬럼만 필터링
auxiliary_cols = [col for col in auxiliary_cols if col in df_features.columns]

# 피처 컬럼
feature_cols = [col for col in df_features.columns 
                if col not in id_cols + date_cols + [target_col] + auxiliary_cols]

print(f"\nID 컬럼: {len(id_cols)}개")
for col in id_cols:
    print(f"  - {col}")

print(f"\n날짜 컬럼: {len(date_cols)}개")
for col in date_cols:
    if col in df_features.columns:
        print(f"  - {col}")

print(f"\nTarget 컬럼: 1개")
print(f"  - {target_col}")

print(f"\n보조 컬럼: {len(auxiliary_cols)}개")
for col in auxiliary_cols[:10]:
    print(f"  - {col}")
if len(auxiliary_cols) > 10:
    print(f"  ... 외 {len(auxiliary_cols)-10}개")

print(f"\n모델링 피처: {len(feature_cols)}개")

# 3. 피처 카테고리별 분류
print("\n" + "-" * 80)
print("피처 카테고리별 분류")
print("-" * 80)

categories = {
    '인구학적': ['age_', 'gender_', 'admit_', 'los_'],
    '진단': ['comorb_'],
    '약물': ['med_', 'gdmt_', 'medication_', 'polypharmacy', 'on_diuretics'],
    '시술': ['proc_', 'procedure_', 'cardiac_intervention', 'respiratory_support', 
             'renal_replacement', 'has_cardiac_device'],
    'ICU': ['icu_'],
    '입원이력': ['past_', 'first_admission', 'frequent_flyer', 'recent_admission', 
                'had_past_readmission', 'days_since_last_admission'],
    'Vital Signs': ['heart_rate', 'blood_pressure', 'respiratory_rate', 'spo2', 
                    'temperature', 'weight', 'height', 'o2_flow', 'fio2', 
                    'pulse_pressure', 'shock_index', 'hypotension', 'tachycardia',
                    'bradycardia', 'hypoxemia', 'tachypnea'],
    'Lab Values': ['lab_', 'egfr', 'bun_cr_ratio', 'hyponatremia', 'hypokalemia',
                   'hyperkalemia', 'anemia', 'leukocytosis', 'bnp_very_high',
                   'ntprobnp_very_high', 'hyperglycemia', 'hypoglycemia', 
                   'hyperlactatemia']
}

category_counts = {}
for category, prefixes in categories.items():
    count = sum(1 for col in feature_cols 
                if any(prefix in col for prefix in prefixes))
    category_counts[category] = count
    print(f"  - {category:<15}: {count:>3}개")

print(f"\n총 모델링 피처: {len(feature_cols)}개")

# 4. 연속형 vs 범주형 구분
print("\n" + "-" * 80)
print("피처 타입 구분")
print("-" * 80)

continuous_features = []
categorical_features = []

for col in feature_cols:
    unique_count = df_features[col].nunique()
    
    if unique_count == 2:
        # 이진형
        categorical_features.append(col)
    elif unique_count <= 10 and df_features[col].dtype in ['int64', 'float64']:
        # 10개 이하 고유값
        categorical_features.append(col)
    elif df_features[col].dtype in ['int64', 'float64']:
        # 연속형
        continuous_features.append(col)
    else:
        # 문자열 등
        categorical_features.append(col)

print(f"연속형 피처: {len(continuous_features)}개")
print("  예시:")
for col in continuous_features[:10]:
    print(f"    - {col}")
if len(continuous_features) > 10:
    print(f"    ... 외 {len(continuous_features)-10}개")

print(f"\n범주형/이진형 피처: {len(categorical_features)}개")
print("  예시:")
for col in categorical_features[:10]:
    print(f"    - {col}")
if len(categorical_features) > 10:
    print(f"    ... 외 {len(categorical_features)-10}개")

In [ ]:
# 결측치 확인
# 대부분 chartevents 파생 피처일 것 (ICU 입원하지 않은 환자는 데이터 X)
missing_summary = []
for col in feature_cols:
    missing_count = df_features[col].isnull().sum()
    if missing_count > 0:
        missing_pct = missing_count / len(df_features) * 100
        missing_summary.append({
            'feature': col,
            'missing_count': missing_count,
            'missing_pct': missing_pct
        })

if len(missing_summary) > 0:
    df_missing = pd.DataFrame(missing_summary)
    df_missing = df_missing.sort_values('missing_count', ascending=False)
    
    print(f"\n결측치 있는 피처: {len(df_missing)}개")
    print("\n상위 20개:")
    for idx, row in df_missing.head(20).iterrows():
        print(f"  - {row['feature']:<50} {row['missing_count']:>6,}건 ({row['missing_pct']:>5.1f}%)")
    
    # 결측치 비율별 분류
    high_missing = (df_missing['missing_pct'] > 50).sum()
    medium_missing = ((df_missing['missing_pct'] > 20) & (df_missing['missing_pct'] <= 50)).sum()
    low_missing = ((df_missing['missing_pct'] > 0) & (df_missing['missing_pct'] <= 20)).sum()
    
    print(f"\n결측치 비율별 분류:")
    print(f"  - >50% 결측: {high_missing}개 (Vital/Lab - ICU/측정 안 함)")
    print(f"  - 20-50% 결측: {medium_missing}개")
    print(f"  - 0-20% 결측: {low_missing}개")
else:
    print("\n✓ 모든 피처에 결측치 없음")

In [ ]:
# 최종 데이터셋 생성 및 저장
# 전체 데이터
final_complete_file = PROCESSED_PATH / 'final_features_complete.csv'
df_features.to_csv(final_complete_file, index=False)

file_size = final_complete_file.stat().st_size / (1024 * 1024)
print(f"\n전체 데이터 저장: {final_complete_file}")
print(f"  - 크기: {file_size:.1f} MB")
print(f"  - 행: {len(df_features):,}건")
print(f"  - 컬럼: {len(df_features.columns)}개")

# 모델링용 데이터 (피처 + Target + hadm_id)
modeling_cols = ['hadm_id'] + feature_cols + [target_col]
df_modeling = df_features[modeling_cols].copy()

modeling_file = PROCESSED_PATH / 'modeling_dataset_final.csv'
df_modeling.to_csv(modeling_file, index=False)

modeling_size = modeling_file.stat().st_size / (1024 * 1024)
print(f"\n모델링용 데이터 저장: {modeling_file}")
print(f"  - 크기: {modeling_size:.1f} MB")
print(f"  - 행: {len(df_modeling):,}건")
print(f"  - 컬럼: {len(modeling_cols)}개")
print(f"    • hadm_id: 1개 (추적용)")
print(f"    • 피처: {len(feature_cols)}개")
print(f"    • Target: 1개 (readmission_30d)")

# 피처 정보 저장
feature_info = pd.DataFrame({
    'feature': feature_cols,
    'dtype': [str(df_features[col].dtype) for col in feature_cols],
    'unique_values': [df_features[col].nunique() for col in feature_cols],
    'missing_count': [df_features[col].isnull().sum() for col in feature_cols],
    'missing_pct': [df_features[col].isnull().sum() / len(df_features) * 100 for col in feature_cols],
    'type': ['continuous' if col in continuous_features else 'categorical' for col in feature_cols]
})

# 카테고리 추가
def get_category(feature_name):
    for category, prefixes in categories.items():
        if any(prefix in feature_name for prefix in prefixes):
            return category
    return '기타'

feature_info['category'] = feature_info['feature'].apply(get_category)

feature_info_file = PROCESSED_PATH / 'feature_info.csv'
feature_info.to_csv(feature_info_file, index=False)

print(f"\n피처 정보 저장: {feature_info_file}")
print(f"  - 피처 메타데이터")
print(f"  - 타입, 결측치, 카테고리 정보")

# D. 연속형/범주형 피처 리스트 저장
feature_types = {
    'continuous_features': continuous_features,
    'categorical_features': categorical_features
}

import json
feature_types_file = PROCESSED_PATH / 'feature_types.json'
with open(feature_types_file, 'w') as f:
    json.dump(feature_types, f, indent=2)

print(f"\n피처 타입 저장: {feature_types_file}")
print(f"  - 연속형: {len(continuous_features)}개")
print(f"  - 범주형: {len(categorical_features)}개")

# 최종 요약

print(f"\n데이터셋 정보:")
print(f"  - 기간: 2008-2019 (MIMIC-IV)")
print(f"  - 대상: 심부전 입원 환자")
print(f"  - 총 입원: {len(df_features):,}건")
print(f"  - 고유 환자: {df_features['subject_id'].nunique():,}명")
print(f"  - 재입원률: {df_features['readmission_30d'].mean()*100:.2f}%")

print(f"\n피처 구성 (총 {len(feature_cols)}개):")
for category, count in category_counts.items():
    print(f"  - {category:<15}: {count:>3}개")

print(f"\n데이터 품질:")
total_values = len(df_features) * len(feature_cols)
total_missing = sum(df_features[col].isnull().sum() for col in feature_cols)
completeness = (1 - total_missing / total_values) * 100
print(f"  - 데이터 완결성: {completeness:.1f}%")
print(f"  - 총 결측치: {total_missing:,}개")

print(f"\n저장된 파일:")
print(f"  ✓ final_features_complete.csv  ({file_size:.1f} MB)")
print(f"  ✓ modeling_dataset.csv         ({modeling_size:.1f} MB) ")
print(f"  ✓ feature_info.csv")
print(f"  ✓ feature_types.json")